In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Device configuration
# training on GPU is faster
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Hyperparameters for controlling the learning process
num_epochs = 5
num_classes = 10
batch_size = 100
learning_rate = 0.001

In [ ]:
# MNIST dataset
train_dataset = torchvision.datasets.MNIST(root='../../data/',
                                           train=True,
                                           transform=transforms.ToTensor(),
                                           download=True)

test_dataset = torchvision.datasets.MNIST(root='../../data/',
                                          train=False,
                                          transform=transforms.ToTensor())

# load the training dataset. Convert images into PyTorch tensors
# transforms.ToTensor() converts pixel values

In [ ]:
# Data loader
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)


In [3]:
# Convolutional neural network (two convolutional layers)
class ConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ConvNet, self).__init__()

        # First convolution block
        self.layer1 = nn.Sequential(
            # Input:28*28*1 -> 28*28*16
            # 16 filters, filter size = 5*5, padding = 2; keeps image size unchanged
            nn.Conv2d(1, 16, kernel_size=5, stride=1, padding=2), #(input channels=1 for grayscale, output channels, kernel, padding) 28*28*1 -> 28*28*16
            nn.BatchNorm2d(16), # normalizes the feature maps
            nn.ReLU(), # applies activation function
            nn.MaxPool2d(kernel_size=2, stride=2)) #reduces image size 28*28 -> 14*14
            # final output = 14*14*16

        # Second convolution block
        self.layer2 = nn.Sequential(
            # Input= 14*14*16 -> 14*14*32
            nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)) # Final output = 7*7*32

        self.fc = nn.Linear(7*7*32, num_classes) # flatten the output so 7*7*32 = 1568

    def forward(self, x):
        out = self.layer1(x)  #28*28*1 -> 14*14*16
        out = self.layer2(out)  #14*14*16 -> 7*7*32
        out = out.reshape(out.size(0), -1) #batch size=100, 100*7*7*32 -> 100*1568
        out = self.fc(out)  #produces 10 output scores
        return out

In [ ]:
model = ConvNet(num_classes).to(device)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()  #combines softmax and negative log likelihood loss
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  #updates the model weights

In [ ]:
# Train the model
total_step = len(train_loader) #stores total number of batches
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)  #prediction step
        loss = criterion(outputs, labels)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()  # computes gradients
        optimizer.step()  # adjusts weights

        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))


In [ ]:
# Test the model
model.eval()  # eval mode
with torch.no_grad():  # disable gradient calculation
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1) # select the index of the class with highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the model on the 10000 test images: {} %'.format(100 * correct / total))

In [ ]:
# Save the model checkpoint
torch.save(model.state_dict(), 'model.ckpt')